In [1]:
import os
import glob
import numpy as np
from music21 import converter, instrument, note, chord
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# ── 1. Parse MIDI files ──────────────────────────────────────────────────────

def parse_midi_files(midi_folder, limit=None):
    """Extract notes and chords from all MIDI files in a folder."""
    notes = []
    files = glob.glob(os.path.join(midi_folder, "**/*.midi"), recursive=True) + \
            glob.glob(os.path.join(midi_folder, "**/*.mid"),  recursive=True)

    if limit:
        files = files[:limit]

    for file in files:
        print(f"Parsing: {file}")
        try:
            midi = converter.parse(file)
            parts = instrument.partitionByInstrument(midi)
            elements = parts.parts[0].recurse() if parts else midi.flat.notes

            for element in elements:
                if isinstance(element, note.Note):
                    notes.append(str(element.pitch))
                elif isinstance(element, chord.Chord):
                    notes.append('.'.join(str(n) for n in element.normalOrder))
        except Exception as e:
            print(f"Skipping {file}: {e}")

    return notes


# ── 2. Encode notes to integers ──────────────────────────────────────────────

def encode_notes(notes):
    """Map each unique note/chord to an integer."""
    le = LabelEncoder()
    encoded = le.fit_transform(notes)
    vocab_size = len(le.classes_)
    print(f"Total notes: {len(notes)} | Vocabulary size: {vocab_size}")
    return encoded, le, vocab_size


# ── 3. Build sequences ───────────────────────────────────────────────────────

def build_sequences(encoded_notes, seq_length=50):
    """
    Slide a window over the encoded notes to create
    (input_sequence, next_note) pairs.
    """
    X, y = [], []
    for i in range(len(encoded_notes) - seq_length):
        X.append(encoded_notes[i : i + seq_length])
        y.append(encoded_notes[i + seq_length])
    return np.array(X), np.array(y)


# ── 4. PyTorch Dataset ───────────────────────────────────────────────────────

class MaestroDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ── 5. Full pipeline ─────────────────────────────────────────────────────────

def preprocess_maestro(midi_folder, seq_length=50, batch_size=64, limit=None):
    # Step 1: Parse
    notes = parse_midi_files(midi_folder, limit=limit)

    # Step 2: Encode
    encoded_notes, label_encoder, vocab_size = encode_notes(notes)

    # Step 3: Sequences
    X, y = build_sequences(encoded_notes, seq_length)
    print(f"X shape: {X.shape} | y shape: {y.shape}")

    # Step 4: DataLoader
    dataset    = MaestroDataset(X, y)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    return dataloader, label_encoder, vocab_size


# ── 6. Run ───────────────────────────────────────────────────────────────────

MIDI_FOLDER = "./maestro-v3.0.0"   # path to your MAESTRO dataset
SEQ_LENGTH  = 50
BATCH_SIZE  = 64

dataloader, label_encoder, vocab_size = preprocess_maestro(
    midi_folder = MIDI_FOLDER,
    seq_length  = SEQ_LENGTH,
    batch_size  = BATCH_SIZE,
    limit       = 10            # set None to use all files
)

print(f"Vocab size: {vocab_size}")
print(f"Batches:    {len(dataloader)}")

C:\Users\Anjaneya Sharma\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_06_Track06_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_08_Track08_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_10_Track10_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_05_R1_2004_01_ORIG_MID--AUDIO_05_R1_2004_02_Track02_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_05_R1_2004_01_ORIG_MID--AUDIO_05_R1_2004_03_Track03_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_05_R1_2004_02-03_ORIG_MID--AUDIO_05_R1_2004_06_Track06_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_07_R1_2004_01_ORIG_MID--AUDIO_07_R1_2004_02_Track02_wav.midi
Parsing: ./maestro-v3.0.0\2004\MIDI-Unprocessed_SMF_07_R1_2004_01_ORIG_MID--AUDIO

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
import torch.nn as nn

class MusicRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, n_layers=2):
        super(MusicRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        rnn_out, hidden = self.rnn(x, hidden)
        output = self.fc(rnn_out[:, -1, :])
        return output, hidden

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MusicRNN(vocab_size=vocab_size, hidden_size=512).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        output, _ = model(x, None)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 5.320256233215332


In [ ]:
from music21 import stream, note, chord
import random

def generate_music(model, label_encoder, seed_notes, length=200, temperature=1.0):
    model.eval()
    generated = list(seed_notes)
    input_seq = torch.tensor([seed_notes], dtype=torch.long).to(device)
    hidden = None

    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            # apply temperature
            probs = torch.softmax(output / temperature, dim=1)
            next_idx = torch.multinomial(probs, 1).item()
            generated.append(next_idx)
            input_seq = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    # convert indices back to note names
    note_names = label_encoder.inverse_transform(generated)

    # build MIDI stream
    midi_stream = stream.Stream()
    for pattern in note_names:
        if '.' in pattern:  # chord
            notes_in_chord = pattern.split('.')
            chord_notes = [note.Note(int(n)) for n in notes_in_chord]
            midi_stream.append(chord.Chord(chord_notes))
        else:
            midi_stream.append(note.Note(pattern))

    midi_stream.write('midi', fp='generated_music.mid')
    print("Saved generated_music.mid — open it with any media player or DAW.")

# use the first 50 notes from the dataset as seed
seed = dataloader.dataset.X[0].tolist()
generate_music(model, label_encoder, seed, length=200, temperature=0.8)